# 04a. Graph Engineering ①: 층 지도, 작업 계약, 전달 계약, 공유 예산

03에서 순차·병렬·분기·루프를 만들었습니다. 그 흐름만으로는 운영에 넣을 수 없습니다. Worker가 무엇을 돌려줘야 하는지, 호출을 어디서 멈출지, 실패하면 무엇을 할지, 환불을 언제 실행할지가 아직 코드에 없기 때문입니다. 04는 두 권으로 나뉩니다. 이 노트북 **04a**는 앞의 두 가지, 곧 Worker가 주고받는 계약과 호출을 멈추는 예산을 그래프에 넣습니다. 실패와 승인은 **04b**에서 이어집니다. 사례는 계속 고객 요청 T-1001입니다. "결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요."

## 학습 목표

- 이 과정의 Graph Engineering 정의를 말하고, 그 정의가 2~8절의 어느 작업에 대응하는지 짚습니다. 2~4절은 이 노트북에, 5~8절은 04b에 있습니다.
- Prompt, Context, Harness, Graph, Loop가 각각 무엇을 설계하는지 이 과정의 부품에 대응시킵니다.
- 역할 프롬프트와 작업 계약의 차이를 Worker 출력으로 비교합니다.
- Worker 사이의 전달 계약을 스키마로 쓰고, 스키마 통과와 근거 검증이 다르다는 것을 확인합니다.
- 요약 전달과 근거 전달의 손실을 **같은 Worker 결과**에서 비교합니다.
- 병렬 Worker가 공유하는 예산을 모델 밖의 카운터 하나로 세고, 넘으면 멈춰서 드러냅니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 층 지도 | 4 |
| 2 | Prompt: 작업 계약 | 6 |
| 3 | Context: 전달 계약과 손실 | 10 |
| 4 | Harness: 공유 예산 | 5 |
| 5 | 직접 해보기와 정리 | 5 |

## 호출 규모

이 노트북은 `Day03_new/.env`의 키로 실제 Claude 모델을 부릅니다. 키 없이 도는 경로는 없습니다. 모델을 부르는 셀은 호출 직전에 `announce`가 규모를 한 줄로 알려 줍니다. 비용은 본인 키로 청구됩니다. 그래프 구조, 검증, 예산처럼 모델의 판단이 필요 없는 부분은 일반 함수로 돌리고 출력에 "모델 호출 0회"를 적습니다. 그림 셀과 앞 실행의 값을 표로 읽기만 하는 셀은 모델을 부르지 않습니다.

| 구분 | 유료 셀 수 | 모델 호출(예고 상한) |
|---|---:|---:|
| 2절 역할 프롬프트 Worker, 작업 계약 Worker | 2 | 4 + 4 |
| 3절 T-1001 Worker 3개 실행, 같은 작성자에게 전달물 3종 주기 | 2 | 12 + 3 |
| 4절 예산 2회 Worker 실행 | 1 | 2 |
| 기본 실행 합계 | 5 | 예고 상한 25 |

`직접 해보기`의 유료 셀 하나(해보기 1)를 켜면 모델 호출이 최대 4회 늘어납니다. 호출 수는 실행마다 달라집니다. 모델이 도구를 몇 번 쓰는지는 모델이 정하기 때문입니다.

In [ ]:
import inspect
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from deepagents import create_deep_agent
from pydantic import ValidationError

from mas_lab.contracts import (FinalAnswer, WorkerResult, WorkerTask, join_results, next_action, plan_hash, trim_payload,
                               validate_result)
from mas_lab.desk import Desk
from mas_lab.evaluation import evaluate_row, load_gold
from mas_lab.hybrid import ROLE_GOALS, ROLES, build_hybrid
from mas_lab.ledger import check_refund
from mas_lab.runner import make_desk
from mas_lab.structures import OUTCOME_GUIDE, scope_line
from mas_lab.trace import ModelMeter, Trace, timeline
from mas_lab.view import show_table
from mas_lab.workers import ROLE_TOOLS, TASK_CONTRACT, allowlist_middleware, contract_prompt, make_worker, run_worker
from rag_lab.evidence import Budget, BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

require_key()  # 키가 없으면 여기서 한국어 안내와 함께 멈춥니다.
print(key_status())
model = make_model()  # 모델 객체를 만드는 것은 호출이 아닙니다.
cases = [json.loads(line) for line in Path("data/cases.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
case = next(c for c in cases if c["id"] == "complex-double-pay")
gold = load_gold()  # 사후 자동 검사에만 씁니다. 도구, 프롬프트, 게이트는 이 값을 보지 않습니다.
print("관통 사례:", case["id"], "|", case["question"])
print("측정 장치:", Desk.__name__, Trace.__name__, ModelMeter.__name__)

## 1. 층 지도: 새로운 이름은 이전 기술을 없애지 않습니다

**문제.** Agent가 틀린 답을 냈을 때 "프롬프트를 고치자"가 첫 반응이 되기 쉽습니다. 그런데 원인이 프롬프트가 아닐 수 있습니다. 필요한 조항이 Worker에게 전달되지 않았을 수도 있고, 재시도가 예산을 다 썼을 수도 있고, 실패한 Worker가 합류에서 조용히 사라졌을 수도 있습니다. 고칠 곳을 고르려면 문제를 층으로 나눠 불러야 합니다.

이 노트북은 다섯 층을 다음처럼 정의합니다. Prompt, Context, Harness 같은 이름은 시기마다 유행처럼 새로 나왔지만, 새 이름이 앞의 일을 없애지 않습니다. 잘 쓴 프롬프트는 Graph가 있어도 여전히 필요합니다.

| 층 | 설계하는 것 | 각 층의 질문 |
|---|---|---|
| Prompt | 지시 | 이 작업에서 무엇을 어떤 조건으로 하라고 말했는가? |
| Context | 제공 정보 | 이 호출에 무엇을 넣고 무엇을 뺐는가? 뺀 것은 어디에 있는가? |
| Harness | 실행 환경 | 어떤 도구를 쓸 수 있는가? 얼마까지 쓸 수 있는가? 누가 검증하고 누가 멈추는가? |
| Graph | 작업·상태·의존관계 | 어떤 일이 어떤 일을 기다리는가? 상태는 어디에 쌓이는가? |
| Loop | 반복·재시도·종료 | 언제 다시 하고, 언제 사람에게 넘기고, 언제 끝내는가? |

**이 과정의 Graph Engineering.** 이 과정에서 Graph Engineering은 작업을 노드와 간선으로 명시하고, 노드마다 누가 판단하는지(코드·모델·사람), 무엇을 주고받는지(계약), 언제 멈추고 다시 하는지(예산·실패별 다음 행동·승인)를 그래프에 적어 넣는 일입니다. 다섯 층 지도의 Graph 층을 설계하면서 나머지 네 층(Prompt·Context·Harness·Loop)을 노드와 간선에 배치하는 작업입니다. 이 노트북의 2~8절이 바로 이것을 층별로 하나씩 만듭니다. 2~4절은 이 노트북(04a)에, 5~8절은 04b에 있습니다.

아래 셀은 이 과정의 실제 부품을 층에 대응시킵니다. 파일과 줄은 `inspect`로 얻으므로 코드가 바뀌면 함께 바뀝니다. 마지막 칸을 읽고, 이 부품을 지웠을 때 어떤 실패가 생길지 먼저 떠올려 보세요.

In [ ]:
def where(obj) -> str:
    # 부품이 정의된 파일과 줄을 코드에서 직접 읽습니다.
    file = Path(inspect.getsourcefile(obj)).resolve().relative_to(Path.cwd().resolve())
    return f"{file}:{inspect.getsourcelines(obj)[1]}"

layers = [("Prompt", contract_prompt, "역할 이름만 받은 Worker가 다음 단계가 쓸 수 없는 출력을 냅니다"),
          ("Context", WorkerResult, "요약만 전달되어 조건과 근거가 사라집니다"),
          ("Context", trim_payload, "바이트로 잘려 예외 문장이 사라집니다"),
          ("Harness", Budget, "호출이 끝없이 늘어나도 멈추지 않습니다"),
          ("Harness", allowlist_middleware, "Worker가 허용되지 않은 도구를 부릅니다"),
          ("Harness", check_refund, "모델의 제안이 곧 실행이 됩니다"),
          ("Graph", join_results, "실패한 Worker가 합류에서 조용히 빠집니다"),
          ("Graph", build_hybrid, "경로를 모델이 매번 새로 정하게 됩니다"),
          ("Loop", next_action, "모든 실패를 똑같이 다시 호출합니다")]
show_table([{"층": layer, "부품": part.__name__, "파일과 줄": where(part), "없으면 생기는 실패": failure}
            for layer, part, failure in layers])
print("모델 호출 0회")

## 2. Prompt: 역할 이름과 작업 계약

**문제.** "너는 장애 보상 전문가야"라고 말하면 전문가처럼 일할까요? 이 지시에는 어떤 값을 근거로 삼을지, 결과를 어떤 칸에 담을지, 자료가 없으면 어떻게 할지, 무엇을 하면 안 되는지가 없습니다. 다음 단계의 코드는 Worker의 말투가 아니라 칸을 읽습니다.

`outage` Worker에게는 가용률 조회 도구만 줍니다. 이 Worker는 기본 이용료도, SLA 크레딧 구간도 볼 수 없습니다. 그래서 아래 작업은 일부러 Worker의 권한을 넘습니다. "2026년 8월 가용률을 확인하고, 그에 따른 서비스 크레딧 금액을 알려 주세요." 볼 수 없는 값을 어떻게 다루는지가 두 지시의 차이를 드러냅니다.

**먼저 예측하세요.** 역할만 있는 지시를 받은 Worker의 출력을 다음 단계의 코드가 그대로 쓸 수 있을까요? 다음 세 가지를 적어 두세요. (1) 크레딧 금액을 지어낼까요, 모른다고 할까요? (2) 근거로 쓸 조회 번호를 답에 남길까요? (3) 코드가 이 작업의 완료 여부를 어느 값으로 판단할 수 있을까요?

In [ ]:
announce("역할 프롬프트 Worker", model_calls=4)
GOAL = "2026년 8월 가용률을 확인하고, 그에 따른 서비스 크레딧 금액을 알려 주세요."
ROLE_PROMPT = "너는 장애 보상 전문가야."
role_desk, role_meter = make_desk(case, label="role"), ModelMeter()
role_agent = create_deep_agent(model=model, tools=role_desk.tools("outage"), system_prompt=ROLE_PROMPT, name="role-outage",
                               middleware=allowlist_middleware(role_desk.budget, ROLE_TOOLS["outage"]))
role_state = role_agent.invoke({"messages": [{"role": "user", "content": GOAL}]},
                               config={"callbacks": [role_meter], "recursion_limit": 40})
role_text = str(role_state["messages"][-1].text)
print("모델 호출:", role_meter.totals()["model_calls"], "| 도구:", role_desk.budget.counts.get("tool", 0))
print(role_text)

In [ ]:
announce("작업 계약 Worker", model_calls=4)
contract_task = WorkerTask(task_id="outage", goal=GOAL, allowed_tools=sorted(ROLE_TOOLS["outage"]))
contract_desk, contract_meter = make_desk(case, label="contract"), ModelMeter()
print(contract_prompt(contract_task, contract_desk.scope))
print("-" * 40)
contract_result = run_worker(contract_task, contract_desk, model, config={"callbacks": [contract_meter]})
print("모델 호출:", contract_meter.totals()["model_calls"], "| 도구:", contract_desk.budget.counts.get("tool", 0))
print(contract_result.model_dump_json(indent=2))

In [ ]:
REF = re.compile(r"\bF-[\w-]+|\bE-[0-9a-f]{12}")
AMOUNT = re.compile(r"\d+\.\d+|\d[\d,]*%|\d{1,3}(?:,\d{3})+")

def amounts(text: str) -> set[str]:
    # 소수, 비율, 쉼표 금액만 봅니다. 날짜의 숫자는 세지 않습니다.
    return {n.replace(",", "").removesuffix("%") for n in AMOUNT.findall(text)}

def source_of_numbers(text: str, desk: Desk) -> str:
    source = {n.replace(",", "") for n in re.findall(r"\d[\d,]*(?:\.\d+)?", desk.source_text())}
    found = amounts(text)
    return f"{len(found & source)}/{len(found)}개가 조회 값에 있음. 없는 값: {sorted(found - source) or '없음'}"

# 자유 텍스트에는 status 칸이 없어서 코드는 complete로 볼 수밖에 없습니다.
role_as_result = WorkerResult(status="complete", facts=[role_text], evidence_refs=REF.findall(role_text))
contract_text = " ".join([*contract_result.facts, *contract_result.uncertainty])
known = {"역할 프롬프트": role_desk.known_refs(), "작업 계약": contract_desk.known_refs()}
show_table([{"지시": "역할 프롬프트", "출력 형태": "자유 텍스트", "status": "칸 없음", "검증 가능한 근거 수": len(set(role_as_result.evidence_refs) & known["역할 프롬프트"]),
             "불확실성": "칸 없음", "사실 수": "칸 없음", "숫자의 출처": source_of_numbers(role_text, role_desk),
             "validate_result": validate_result(role_as_result, known["역할 프롬프트"]) or "문제 없음"},
            {"지시": "작업 계약", "출력 형태": "WorkerResult", "status": contract_result.status, "검증 가능한 근거 수": len(set(contract_result.evidence_refs) & known["작업 계약"]),
             "불확실성": len(contract_result.uncertainty), "사실 수": len(contract_result.facts), "숫자의 출처": source_of_numbers(contract_text, contract_desk),
             "validate_result": validate_result(contract_result, known["작업 계약"]) or "문제 없음"}])
print("모델 호출 0회")

**관찰할 것.** 표에서 `출력 형태`, `status`, `불확실성` 칸을 먼저 읽으세요. 역할 프롬프트 Worker의 답이 사람에게는 충분히 좋아 보일 수 있습니다. 모른다는 말을 정직하게 쓰고 조회 번호까지 적었을 수도 있습니다. 그렇더라도 그 정직함은 문장 안에 있습니다. 코드는 문장을 읽어 완료인지 불확실한지 판단할 수 없습니다. 작업 계약 Worker는 같은 말을 `status`와 `uncertainty` 칸에 담습니다.

`숫자의 출처` 칸은 다르게 읽습니다. 조회 값에 없는 숫자가 있다면 그 숫자가 어디서 왔는지 출력된 문장에서 직접 찾으세요. 금액을 단정하지 않고 "예를 들어"라며 일반적인 구간을 적어 둔 것일 수도 있습니다. 사람은 예시로 읽지만 다음 단계의 코드는 구별하지 못합니다. 없는 값이 없다면 이번 실행에서는 두 지시 모두 숫자를 지어내지 않은 것입니다. 이 경우에도 계약의 효과가 없다는 뜻은 아닙니다. 계약은 평균 품질이 아니라 다음 단계가 믿고 읽을 칸을 정합니다.

`validate_result` 칸도 읽으세요. 역할 프롬프트 행은 status가 없어 코드가 complete로 가정하고 넣은 결과입니다. 작업 계약 행에 `등록되지 않은 근거`가 나왔다면 계약을 받은 모델도 사건 번호처럼 조회 참조가 아닌 값을 근거 칸에 넣을 수 있다는 뜻입니다. 계약은 어길 수 없는 규칙이 아니라 지시입니다. 그래서 다음 절에서 코드로 검사합니다.

역할 프롬프트가 무의미하다는 뜻이 아닙니다. 역할은 말투와 관점을 정하는 데 쓸모가 있습니다. 다만 역할 이름만으로 업무가 완성되지는 않습니다. 그리고 이것은 1회 관찰입니다. 같은 지시를 다시 돌리면 답이 달라질 수 있어 이 한 번으로 어느 쪽이 얼마나 낫다고 말할 수 없습니다. 차이가 어떤 칸에서 생기는지를 읽는 데에만 쓰세요.

## 3. Context: 전달 계약과 손실

**문제.** Worker 사이에 무엇을 넘기느냐가 다음 단계가 볼 수 있는 세계를 정합니다. 2절의 계약이 Worker 한 명에게 일을 시키는 약속이라면, 이번 절의 계약은 Worker와 Worker, Worker와 작성자 사이의 약속입니다. 입력 계약 `WorkerTask`는 일을 나눠 보내는 쪽(이 과정에서는 그래프의 분배 노드)이 Worker에게 주는 것이고, 반환 계약 `WorkerResult`는 Worker가 돌려주는 것입니다.

입력에 `allowed_tools`가 있는 이유를 스키마를 보며 생각해 보세요. 작업마다 필요한 권한을 작업 자체에 적어 두면 무엇을 볼 수 있고 무엇을 할 수 있는지를 실행 전에 검토할 수 있습니다. 결과를 받은 쪽도 이 Worker가 원래 볼 수 없는 사실을 어떻게 알았는지 물을 수 있습니다. 반환 계약이 요약 대신 `facts`, `evidence_refs`, `uncertainty`를 따로 두는 이유는 다음 단계가 사실과 근거를 대조하고 모르는 것을 물려받게 하기 위해서입니다.

In [ ]:
def schema_rows(contract) -> list[dict]:
    schema = contract.model_json_schema()
    return [{"계약": contract.__name__, "필드": name, "타입": spec.get("type", "object"), "허용 값": spec.get("enum", ""),
             "필수": name in schema.get("required", []), "기본값": spec.get("default", "")}
            for name, spec in schema["properties"].items()]

show_table(schema_rows(WorkerTask) + schema_rows(WorkerResult))
print("allowed_tools를 작업에 두면 이 작업의 권한을 실행 전에 검토할 수 있습니다.")
print("모델 호출 0회")

In [ ]:
# 실패 재현: 잘못된 Worker 결과 네 개를 스키마와 근거 검증에 차례로 넣습니다.
check_desk = make_desk(case, label="check")
check_desk.tools("outage")[0].invoke({"month": "2026-08"})  # 실제 조회로 F-AV-2026-08을 근거로 등록합니다.
refs = check_desk.known_refs()
bad_results = {"없는 status": {"status": "done", "facts": ["가용률 99.3"], "evidence_refs": ["F-AV-2026-08"]},
               "없는 근거 ID": {"status": "complete", "facts": ["가용률 99.3"], "evidence_refs": ["lookup_outage: 8월 장애"]},
               "근거 없는 complete": {"status": "complete", "facts": ["가용률 99.3"], "evidence_refs": []},
               "실제 근거에 틀린 사실": {"status": "complete", "facts": ["가용률 99.95"], "evidence_refs": ["F-AV-2026-08"]}}
rows = []
for name, data in bad_results.items():
    try:
        result, schema = WorkerResult(**data), "통과"
    except ValidationError as error:
        result, schema = None, f"거절: {error.errors()[0]['type']}"
    problems = validate_result(result, refs) if result else None
    rows.append({"잘못된 결과": name, "Pydantic 스키마": schema, "validate_result": "검사 전에 멈춤" if result is None else problems or "통과",
                 "거절한 층": "스키마" if result is None else "근거 검증" if problems else "어느 층도 거절하지 않음"})
show_table(rows)
print("등록된 근거:", sorted(refs), "| 모델 호출 0회")

표의 위 세 행은 각각 다른 층이 거절합니다. 스키마는 모양을 보고, `validate_result`는 이번 실행에 등록된 근거인지를 봅니다. 두 번째 행의 `"lookup_outage: 8월 장애"`는 지어낸 예가 아닙니다. 이 과정을 준비하며 돌린 시험 실행에서 실제 모델이 `evidence_refs`에 이런 값을 냈습니다. 문자열 칸이므로 스키마는 통과합니다.

네 번째 행을 주의해서 읽으세요. 근거 ID는 진짜이고 사실은 틀렸습니다. 검증을 두 겹 두어도 이 행은 지나갑니다. 근거 검증은 인용이 이번 실행에 있었는지를 볼 뿐, 문장이 그 근거와 뜻이 같은지를 판단하지 않습니다. 그래서 인용이 있는 결과라도 사람이 원문과 대조하는 일이 남습니다.

이제 전달 형태로 넘어갑니다. 투자 심사 Agent를 설계하던 판서에 이런 메모가 있습니다. (원본은 저장소의 `Day03/Day03_01.png`입니다. 용량이 커서 이 폴더에 복사하지 않고 말로 옮겼습니다.) "개별 에이전트가 핵심 1줄만 → 80점", "Context가 넘쳐나는데 어떻게 요약할 것인가?" 요약은 입력을 줄여 주지만 무엇을 지우는지 말해 주지 않습니다. 이 메모를 T-1001에서 직접 재 봅니다.

**먼저 예측하세요.** T-1001의 크레딧은 기본 이용료를 기준으로 계산하고(`sla-a-v2.md:11`), 신청 기한은 추가 약정이 정합니다(`sla-a-amendment.md:6`). Worker마다 핵심 1줄만 작성자에게 넘기면 무엇이 먼저 사라질까요? 1,200,000원을 계산 기준으로 잘못 쓸까요, 아니면 금액을 모른다고 할까요? 신청 기한은 며칠로 답할까요? 답을 적은 뒤 실행하세요.

### 그림으로 보기: 잘못된 결과를 어느 층이 거절하는가

위 표의 `거절한 층` 열을 그림으로 옮겼습니다. 네 결과가 왼쪽에서 오른쪽으로 두 검사를 차례로 지납니다.

![잘못된 Worker 결과 네 개가 Pydantic 스키마와 validate_result를 차례로 지난다. 없는 status는 스키마에서, 없는 근거 ID와 근거 없는 complete는 validate_result에서 거절되고, 실제 근거에 틀린 사실은 어느 검사도 거절하지 않고 통과한다](./assets/04a-1-check-layers.png)

- 스키마는 모양을 봅니다. `status`가 정해진 값이 아니면 여기서 멈춥니다.
- `validate_result`는 이번 실행에 등록된 근거인지를 봅니다. 문자열 칸이라 스키마를 통과한 가짜 근거 ID를 여기서 잡습니다.
- 마지막 행은 두 검사를 모두 지납니다. 거르지 못한 틀림은 사람이 원문과 대조해야 합니다.

In [ ]:
announce("T-1001 Worker 3개 실행", model_calls=12)
case_desk, case_meter = make_desk(case, label="payloads"), ModelMeter()

def run_role(role: str) -> tuple[str, WorkerResult]:
    task = WorkerTask(task_id=role, goal=f"{ROLE_GOALS[role]}\n고객 요청: {case['question']}", allowed_tools=sorted(ROLE_TOOLS[role]))
    return role, run_worker(task, case_desk, model, config={"callbacks": [case_meter]})

with ThreadPoolExecutor(max_workers=3) as pool:
    worker_results = dict(pool.map(run_role, ROLES))
# 뒤 셀들은 이 결과를 값으로 읽습니다. 예산의 시간 한도가 뒤 셀까지 이어지지 않게 Desk를 다시 쓰지 않습니다.
case_refs, case_evidence = case_desk.known_refs(), dict(case_desk.context.evidence)
case_quotes = {**dict(case_desk.fact_refs), **{ref: item.quote for ref, item in case_evidence.items()}}
results_hash = plan_hash({role: result.model_dump() for role, result in worker_results.items()})
show_table([{"작업": role, "status": r.status, "사실 수": len(r.facts), "근거 수": len(r.evidence_refs), "불확실성 수": len(r.uncertainty),
             "검증 문제": validate_result(r, case_refs) or "없음", "오류": r.error} for role, r in worker_results.items()])
print(timeline(case_desk.trace.spans))
print("모델 호출:", case_meter.totals()["model_calls"], "| Worker 결과 지문:", results_hash)

In [ ]:
def quotes_of(result: WorkerResult) -> dict[str, str]:
    return {ref: case_quotes[ref] for ref in result.evidence_refs if ref in case_quotes}

def summary_line(result: WorkerResult) -> str:
    # 코드가 만듭니다. 요약에 모델을 쓰면 비교할 변수가 하나 더 늘어납니다.
    return result.facts[0] if result.facts else f"(사실 없음: {result.status})"

evidence_shape = {role: r.model_dump(include={"status", "facts", "evidence_refs", "uncertainty", "error"}) for role, r in worker_results.items()}
payloads = {"1줄 요약": {role: summary_line(r) for role, r in worker_results.items()},
            "사실+근거+불확실성": evidence_shape,
            "원문 포함": {role: {**evidence_shape[role], "quotes": quotes_of(r)} for role, r in worker_results.items()}}
payload_text = {name: json.dumps(p, ensure_ascii=False, indent=1) for name, p in payloads.items()}
show_table([{"전달물": name, "바이트": len(text.encode("utf-8")), "인용 근거 참조": len(set(REF.findall(text))),
             "불확실성 칸": "uncertainty" in text, "Worker 결과 지문": results_hash} for name, text in payload_text.items()])
print(payload_text["1줄 요약"])
print("모델 호출 0회. Worker를 다시 돌리지 않습니다. 전달 형태 하나만 바꾸기 위해서입니다.")

In [ ]:
announce("같은 작성자에게 전달물 3종 주기", model_calls=3)
WRITER_PROMPT = ("고객에게 보낼 답을 씁니다. {scope}\n{guide}\n"
                 "환불이 맞다면 proposal에 action, payment_id, amount, reason을 쓰세요. 실행 여부는 코드 게이트가 정합니다.\n"
                 "아래 전달물이 당신이 가진 자료의 전부입니다. 보상 금액은 전달물로 계산할 수 있으면 계산하고, 확인되지 않는 것은 uncertainty에 쓰세요.\n"
                 "evidence_refs에는 전달물에 적힌 fact_ref와 evidence_id만 넣으세요.\n고객 요청: {question}\n전달물:\n{payload}")
guide = "\n".join(OUTCOME_GUIDE.splitlines()[:3])
writer, writer_meter, writer_rows = model.with_structured_output(FinalAnswer, method="json_schema"), ModelMeter(), []
for name, text in payload_text.items():
    prompt = WRITER_PROMPT.format(scope=scope_line(case_desk), guide=guide, question=case["question"], payload=text)
    try:
        final, error = writer.invoke(prompt, config={"callbacks": [writer_meter]}), ""
    except Exception as failure:  # 실패한 호출도 표에 행으로 남깁니다.
        final, error = None, type(failure).__name__
    writer_rows.append({"전달물": name, "final": final, "오류": error, "입력 토큰": writer_meter.calls[-1]["input_tokens"], "지문": results_hash})
    print(f"--- {name}\n{final.text if final else error}\n불확실성: {final.uncertainty if final else ''}")
print("모델 호출:", writer_meter.totals()["model_calls"], "| usage 누락:", writer_meter.totals()["usage_missing"])

In [ ]:
def writer_row(item: dict) -> dict:
    final = item["final"]
    refs = final.evidence_refs if final else []
    return {"structure": item["전달물"], "case_id": case["id"], "answer": final.text if final else "", "uncertainty": final.uncertainty if final else [],
            "outcome": final.outcome if final else None, "cited_docs": sorted({case_evidence[r].doc_id for r in refs if r in case_evidence}),
            "ledger_refunds": 0, "status": "ok" if final else "model_error"}

CONDITION = re.compile(r"계산 기준|포함하지 않|(?:사용량|사용료).{0,30}제외")
DEADLINE_20, DEADLINE_15 = re.compile(r"(?<!\d)20일|09-20"), re.compile(r"(?<!\d)15일|09-15")
checked, condition_rows = [], []
for item in writer_rows:
    row, final = writer_row(item), item["final"]
    check = evaluate_row(row, gold)
    refs = final.evidence_refs if final else []
    answer, given = row["answer"] + " ".join(row["uncertainty"]), payload_text[item["전달물"]]
    checked.append({"전달물": item["전달물"], "입력 토큰": item["입력 토큰"], "자동 검사": check["자동 검사"], "outcome": check["실제 outcome"], "누락 사실": check["누락 사실"],
                    "금지 포함": check["금지 포함"], "누락 문서": check["누락 문서"], "유효 인용": sum(r in case_refs for r in refs), "없는 인용": [r for r in refs if r not in case_refs]})
    condition_rows.append({"전달물": item["전달물"], "전달물에 계산 기준 조건": bool(CONDITION.search(given)), "답에 계산 기준 조건": bool(CONDITION.search(answer)), "답에 150,000": "150,000" in answer or "150000" in answer,
                           "전달물에 20일 기한": bool(DEADLINE_20.search(given)), "답에 20일 기한": bool(DEADLINE_20.search(answer)), "답에 15일 기한": bool(DEADLINE_15.search(answer))})
show_table(checked)
show_table(condition_rows)
print("자동 검사는 문자열 포함 검사입니다. 표현이 다르면 놓칠 수 있으니 위에 출력된 답과 함께 읽으세요. 모델 호출 0회")

In [ ]:
policy_result = worker_results["policy"]
policy_quotes = quotes_of(policy_result)
full_size = trim_payload(policy_result, policy_quotes, 10**9)["bytes"]
rows = []
for limit in (full_size, int(full_size * 0.6), int(full_size * 0.3), 120):
    trimmed = trim_payload(policy_result, policy_quotes, limit)
    kept = trimmed["payload"]
    rows.append({"예산 바이트": limit, "남은 바이트": trimmed["bytes"], "예산 초과": trimmed["over_budget"], "뺀 항목 수": len(trimmed["dropped"]),
                 "뺀 항목": trimmed["dropped"][:3], "남은 사실": len(kept["facts"]), "남은 원문": len(kept["quotes"]),
                 "남은 근거 참조": len(kept["evidence_refs"]), "불확실성 유지": kept["uncertainty"] == policy_result.uncertainty})
show_table(rows)
print("줄이는 순서는 원문, 그다음 사실입니다. 근거 참조와 불확실성은 끝까지 남깁니다. 모델 호출 0회")

In [ ]:
# 실패 재현: 바이트에서 그냥 자르면 예외 문장이 사라집니다.
cut_desk = make_desk(case, label="cut")
clause = cut_desk.context.read_more("sla-a-v2", 12, 3)["evidence"]["quote"]  # sla-a-v2.md 12~14행의 제외 시간 조항입니다.
EXCEPTION = "단, 월 누적 120분을 초과한 점검 시간은 포함한다"
raw = clause.encode("utf-8")
rows = []
for limit in (len(raw), 180, 120, 100, 60):
    kept = raw[:limit].decode("utf-8", errors="ignore")
    rows.append({"예산 바이트": limit, "예외 문장 남음": EXCEPTION in kept, "남은 끝부분": kept[-34:].replace("\n", " ")})
show_table(rows)

# 검색에서 빠진 문서는 뒤에서 되살릴 수 없습니다. 줄이는 쪽은 받은 것만 줄일 수 있습니다.
query = "서비스 크레딧 신청 기한"
shown = [{"검색 limit": limit, "찾은 문서": sorted({e["doc_id"] for e in search["evidence"]}), "찾지 못한 관련 문서": search["related_docs"]}
         for limit in (1, 6) for search in [make_desk(case, label="search").context.search(query, limit=limit)]]
show_table(shown)
print("모델 호출 0회")

**관찰할 것.** 두 가지 수를 구분해서 읽으세요. `입력 토큰`은 작성자 호출 전체의 입력이고, 전달물의 `바이트`는 그중 근거 Context의 크기입니다. 지시문과 스키마는 세 호출이 같으므로 토큰 차이는 전달물에서 옵니다. 토큰이 줄었는데 `누락 사실`이나 `누락 문서`가 늘었다면 두 변화를 항상 함께 보고하세요. 토큰이 줄었다는 절반만 보고하면 손해를 이득으로 잘못 적게 됩니다.

`자동 검사` 표는 1줄 요약 행부터 읽습니다. `금지 포함`에 `180000`이 있으면 사용량 초과 제외 조건이 사라진 것이고, `누락 사실`에 `150000`이 있고 금지 표현이 없다면 작성자가 금액을 정직하게 비워 둔 것입니다. 답이 그럴듯해도 `유효 인용`이 0이면 다음 단계가 대조할 근거가 없습니다. `없는 인용` 칸에 값이 있다면 값을 직접 읽으세요. 1줄 요약에는 참조가 하나도 없는데도 전달물의 키 이름(`billing`)을 근거처럼 적을 수 있습니다. 다른 행에서는 Worker가 근거 칸에 넣은 사건 번호(`INC-311`)가 작성자의 인용까지 그대로 흘러갈 수 있습니다. Worker 표의 `검증 문제` 칸에 같은 값이 있는지 대조해 보세요. 검증 결과를 보고만 넘기면 잘못된 참조가 다음 단계로 전달됩니다. 어디서 걸러야 할지는 04b의 6절 합류에서 다시 생각합니다.

두 번째 표에서 `전달물에`로 시작하는 칸이 거짓이면 전달 전에 이미 조건을 잃은 것이고, 참인데 `답에` 칸이 거짓이면 전달은 됐지만 작성자가 쓰지 않은 것입니다. 원인이 다르므로 고칠 곳도 다릅니다. 이 표도 문자열 검사이므로 문장의 뜻은 위에 출력된 답을 직접 읽어 확인하세요.

두 표가 어긋나는 행도 찾아보세요. `누락 사실`에 `9월20일`이 있는데 `답에 20일 기한`이 참이라면, 답이 `2026-09-20`처럼 다른 표기로 쓴 것입니다. 답은 맞는데 자동 검사가 놓친 경우입니다. 그래서 칸 이름이 `정답`이 아니라 `자동 검사`입니다. 이 칸의 거짓을 바로 오답으로 옮겨 적지 마세요.

사실+근거+불확실성 행과 원문 포함 행의 `자동 검사`가 다르면 먼저 그 차이가 표기 때문인지 확인하세요. 표기 차이를 빼고 결과가 같다면, 원문을 더 넘긴 입력 토큰이 이번 1회에서는 필수 사실을 늘리지 못한 것입니다. 그렇다고 원문이 항상 낭비는 아닙니다. Worker가 사실을 잘못 옮겼을 때 작성자가 대조할 수 있는 유일한 자료이기 때문입니다. 세 행의 결과가 모두 같게 나왔다면 Worker가 첫 번째 사실에 핵심을 많이 몰아 썼을 수 있습니다. 이 경우 "요약이 괜찮다"가 아니라 "이번 1회에서는 잃지 않았다"로 기록하세요.

이 작성자에게는 계산 도구가 없습니다. 판서의 다른 질문이 여기로 이어집니다. "숫자가 들어가면 LLM은 거짓말을 할 가능성이 몇 %?", "Tool: 계산기". 곱셈은 코드에게 맡기고 비율을 고르는 판단만 모델이 하는 것이 04b 8절의 `calculate_credit`입니다.

`trim_payload` 표에서는 `남은 원문`이 먼저 줄고 `남은 사실`이 뒤에 줄어듭니다. 근거 참조와 불확실성은 끝까지 유지합니다. 이 순서는 `policy` Worker가 근거 참조를 돌려줬을 때만 보입니다. Worker 표에서 `policy`의 `근거 수`가 0이면 줄일 원문과 근거 참조가 처음부터 없습니다. `complete`였다면 `검증 문제`에 `근거 없이 complete입니다.`도 함께 나옵니다. 그 실행에서는 `남은 원문`과 `남은 근거 참조`가 모든 행에서 0이고 `남은 사실`만 줄어드니, 원문이 먼저 빠지는 순서를 이 표로 확인했다고 적지 마세요. 가장 작은 예산에서 `예산 초과`가 참이면 근거 참조와 불확실성이 이미 예산보다 크다는 뜻입니다. 빠뜨리지 않고 넘겼다고 알려 주는 것이지 예산을 지킨 것은 아닙니다. 항목 단위로 줄이면 `뺀 항목` 칸에 무엇을 뺐는지 남습니다.

바이트 자르기 표는 다릅니다. `sla-a-v2.md:13`의 "단, 월 누적 120분을 초과한 점검 시간은 포함한다"가 사라져도 남은 문장은 멀쩡해 보입니다. 뜻이 바뀌었는데 오류가 나지 않습니다. 줄이는 쪽은 받은 것 안에서만 일할 수 있습니다. 검색 표의 `limit=1` 행처럼 추가 약정이 처음부터 들어오지 않았다면 어떤 요약도 그 약정을 복원할 수 없습니다. 이 손실은 검색 설계로 돌아가 고쳐야 합니다.

### 그림으로 보기: 전달물에 무엇이 들어가고 무엇이 먼저 빠지는가

위 표의 세 전달물과 바로 아래 `trim_payload` 표를 한 장으로 겹쳐 봅니다. 위는 세 전달물에 들어가는 칸이고, 아래는 예산이 모자랄 때 줄이는 순서입니다.

![세 전달물에 들어가는 칸. 1줄 요약은 첫 사실만, 사실+근거+불확실성은 status, facts, evidence_refs, uncertainty, error를, 원문 포함은 거기에 quotes를 더한다. trim_payload는 quotes, facts 순으로 빼고 evidence_refs와 uncertainty는 끝까지 남긴다](./assets/04a-2-payload-fields.png)

- 1줄 요약은 `uncertainty` 칸이 아예 없습니다. 모르는 것을 물려받을 자리가 없다는 뜻입니다.
- 줄이는 순서는 코드(`trim_payload`)가 정합니다. 근거 참조와 불확실성은 예산이 아무리 작아도 남깁니다.

## 4. Harness: 공유 예산

**문제.** Worker 셋이 각자 호출 한도를 가지면 전체 비용은 누가 막을까요? 재시도와 병렬이 더해지면 호출 수는 쉽게 늘어납니다. 모델은 자기가 얼마를 썼는지 모르고, 알더라도 멈춰 달라는 지시는 강제력이 없습니다. 실행, 검증, 중단, 복구는 모델 주변 시스템의 책임입니다.

`Budget`은 한 실행의 호출을 하나의 카운터로 셉니다. 먼저 모델 없이 동작만 확인합니다. 스레드 3개가 각 3번씩 소모를 시도하고 한도는 5입니다. 표의 `누적 model`이 한도를 넘는 행이 있는지 찾아보세요. 잠금이 있어 여러 스레드가 동시에 세어도 한도를 넘지 않아야 합니다.

In [ ]:
shared = Budget(max_model_calls=5)
log, log_lock = [], threading.Lock()

def consumer(name: str) -> None:
    for attempt in range(1, 4):
        time.sleep(0.01 * attempt)
        try:
            shared.consume("model", worker=name)
            outcome = "허용"
        except BudgetExceeded as error:
            outcome = f"BudgetExceeded: {error}"
        with log_lock:
            log.append({"순서": len(log) + 1, "스레드": name, "시도": attempt, "결과": outcome, "누적 model": shared.counts.get("model", 0)})

threads = [threading.Thread(target=consumer, args=(name,)) for name in ("billing", "policy", "outage")]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
show_table(log)
print("허용:", sum(r["결과"] == "허용" for r in log), "| 거절:", sum(r["결과"] != "허용" for r in log), "| 모델 호출 0회")

In [ ]:
# 실패 재현: 모델 호출 예산 2회로 Worker 셋을 차례로 실행합니다.
announce("예산 2회 Worker 실행", model_calls=2)
small_desk, small_meter = make_desk(case, budget={"max_model_calls": 2}, label="small"), ModelMeter()
rows, stopped = [], False
for role in ROLES:
    task = WorkerTask(task_id=role, goal=f"{ROLE_GOALS[role]}\n고객 요청: {case['question']}")
    if stopped:
        rows.append({"작업": role, "상태": "시작하지 않음", "오류": "앞 작업에서 예산 소진", "누적 model": small_desk.budget.counts.get("model", 0)})
        continue
    try:
        status, error = run_worker(task, small_desk, model, config={"callbacks": [small_meter]}).status, ""
    except BudgetExceeded as failure:
        status, error, stopped = "budget_exceeded", str(failure), True
    rows.append({"작업": role, "상태": status, "오류": error, "누적 model": small_desk.budget.counts.get("model", 0)})
show_table(rows)
report = small_desk.budget.report()
print("budget.report() counts:", report["counts"], "| 경과 초:", report["elapsed_seconds"], "| 실제 모델 호출:", small_meter.totals()["model_calls"])
show_table([{k: e.get(k, "") for k in ("event", "seconds", "name")} for e in report["events"] if e["event"] != "usage"])

**관찰할 것.** 첫 번째 표에서 한도를 넘는 `누적 model`이 없는지 확인하세요. 어떤 스레드가 거절당하는지는 실행마다 달라질 수 있지만, 허용 건수는 한도와 같아야 합니다. 두 번째 표는 `상태` 칸을 읽습니다. 예산을 다 쓴 작업이 어느 것인지, 그 앞 작업의 결과는 살아 있는지 보세요. `budget_exceeded`는 조용히 넘어갈 일이 아니라 드러내야 할 실패입니다. 셋째 표는 예산을 어떤 호출이 소진했는지 시간 순서로 보여 줍니다.

여기서 할 수 있는 가장 나쁜 대응은 숫자를 올려 실패를 감추는 것입니다. 그 전에 물어야 합니다. 이 호출들은 다 필요했는가? 같은 도구를 반복해 부르지는 않았는가? 한 Worker가 다른 Worker의 예산까지 먹었는가? 근거를 갖고 한도를 정했다면 그때 한도를 바꾸세요. 그 이유도 함께 기록합니다.

### 그림으로 보기: 하나의 예산이 세는 방식

위 표의 실험을 그림으로 옮겼습니다. 스레드 셋은 각자 자기 한도를 갖지 않고 `Budget` 하나에 소모를 요청합니다.

![billing, policy, outage 스레드가 잠금이 있는 Budget 카운터 하나에 소모를 요청한다. 한도 5 이하면 허용되고 한도를 넘으면 BudgetExceeded가 난다](./assets/04a-3-shared-budget.png)

- 한도를 Worker마다 따로 두면 전체 비용은 누구도 막지 않습니다. 하나로 세야 합니다.
- 어느 스레드가 거절당하는지는 실행마다 다릅니다. 같아야 하는 것은 허용 건수가 한도와 같다는 점입니다.

## 직접 해보기

두 칸 모두 조건 하나만 바꿉니다. 먼저 무엇이 달라질지 적은 뒤 돌리세요. 해보기 1만 모델을 부르므로 `RUN_EXERCISE`를 켜야 합니다. 해보기 2는 모델을 부르지 않고 바로 실행됩니다. 실패와 승인을 바꿔 보는 해보기는 04b에 있습니다.

### 해보기 1

작업 계약의 한 줄(한계 또는 권한)을 지우고 2절의 계약 Worker를 다시 돌립니다. 달라지는 칸을 찾으세요. 달라지지 않았다면 1회 관찰로 무엇을 말할 수 있는지도 적습니다.

In [ ]:
RUN_EXERCISE = False
REMOVE_LINE = "한계:"  # "권한:"으로 바꿔 볼 수 있습니다.
if RUN_EXERCISE:
    announce("해보기 1: 계약 한 줄 지우기", model_calls=4)
    contract = "\n".join(line for line in TASK_CONTRACT.splitlines() if not line.startswith(REMOVE_LINE))
    exercise_desk = make_desk(case, label="exercise-1")
    content = contract.format(goal=GOAL, entity=exercise_desk.scope.entity, as_of=exercise_desk.scope.as_of.isoformat(), facts="없음")
    state = make_worker("outage", exercise_desk, model).invoke({"messages": [{"role": "user", "content": content}]}, config={"recursion_limit": 40})
    answer = state["structured_response"]
    print(content)
    show_table([{"지시": name, "status": r.status, "사실 수": len(r.facts), "근거 수": len(r.evidence_refs), "불확실성 수": len(r.uncertainty),
                 "숫자의 출처": source_of_numbers(" ".join([*r.facts, *r.uncertainty]), d)}
                for name, r, d in (("전체 계약", contract_result, contract_desk), (f"{REMOVE_LINE} 삭제", answer, exercise_desk))])
else:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")

### 해보기 2

`trim_payload`는 원문을 먼저, 사실을 나중에 뺍니다. 아래 `DROP_ORDER`를 바꿔 어떤 항목이 먼저 사라지게 할지 정하세요. `uncertainty`를 먼저 지우면 작성자가 무엇을 모른 채 답하게 될까요? 그 선택이 어떤 답을 위험하게 만드는지 적습니다.

In [ ]:
DROP_ORDER = ["uncertainty", "facts", "quotes"]  # 먼저 버릴 칸부터 적습니다.
MAX_BYTES = int(full_size * 0.5)

def my_trim(result: WorkerResult, quotes: dict[str, str], max_bytes: int, order: list[str]) -> dict:
    payload = {"facts": list(result.facts), "evidence_refs": list(result.evidence_refs), "uncertainty": list(result.uncertainty), "quotes": dict(quotes)}
    size = lambda: len(json.dumps(payload, ensure_ascii=False).encode("utf-8"))
    dropped = []
    for field in order:
        while size() > max_bytes and payload[field]:
            item = payload[field].popitem()[0] if isinstance(payload[field], dict) else payload[field].pop()
            dropped.append(f"{field}: {item[:20]}")
    return {"payload": payload, "bytes": size(), "dropped": dropped}

mine, standard = my_trim(policy_result, policy_quotes, MAX_BYTES, DROP_ORDER), trim_payload(policy_result, policy_quotes, MAX_BYTES)
show_table([{"순서": name, "바이트": t["bytes"], "남은 사실": len(t["payload"]["facts"]), "남은 원문": len(t["payload"]["quotes"]),
             "남은 불확실성": len(t["payload"]["uncertainty"]), "뺀 항목 수": len(t["dropped"])}
            for name, t in (("내 순서", mine), ("trim_payload", standard))])
print("내 순서로 뺀 항목:", mine["dropped"], "| 모델 호출 0회")

## 정리

이 노트북에서 그래프에 넣은 것을 층별로 다시 적습니다.

- **Prompt.** 역할 이름이 아니라 작업, 근거, 출력, 한계, 권한을 적은 계약으로 Worker를 정의했습니다. 계약은 평균 품질이 아니라 다음 단계가 읽을 칸을 정합니다.
- **Context.** 사실, 근거, 불확실성을 칸으로 나눠 전달했습니다. 스키마 통과와 근거 검증은 다른 검사이며, 그 둘을 지나도 뜻이 맞다는 보장은 없습니다. 줄이는 일은 항목 단위로 하고, 뺀 것을 기록합니다.
- **Harness.** 예산을 모델 밖에서 하나로 세고, 넘으면 멈춰서 드러냈습니다. 숫자를 올려 실패를 감추기 전에 그 호출이 모두 필요했는지부터 묻습니다.

남은 두 층, Graph와 Loop는 04b에서 다룹니다. 실패 종류에 따라 다음 행동을 나누고, 부분 실패를 합류 노드에서 드러내고, 승인과 멱등을 그래프에 넣습니다.

**읽을거리.** 이 과정은 1절에서 적은 정의를 씁니다. "Graph Engineering"은 2026년에 퍼진 외부 용어이고 출처마다 범위가 다릅니다. 관심이 있으면 다음 두 글을 비교용으로 읽어 보세요. 어느 쪽 정의가 옳은지보다, 두 글이 각각 무엇을 그래프의 일로 보는지를 위 층 지도에 대응시켜 보는 것을 권합니다.

- https://towardsdatascience.com/graph-engineering-for-ai-agents-from-prompts-and-loops-to-workflows/
- https://rickhigh.substack.com/p/graph-engineering-is-two-ideas-wearing

## 확인 질문

1. 인용 ID가 있고 스키마를 통과한 `WorkerResult`도 왜 신뢰할 수 없습니까? 3절의 잘못된 결과 표에서 어느 행이 근거입니까?
2. 3절의 전달물 비교가 전달 형태의 효과를 보이기 위해 Worker를 다시 돌리지 않은 이유는 무엇입니까? 다시 돌렸다면 무엇과 섞입니까?
3. Worker 셋이 각자 호출 한도를 갖는 것과 `Budget` 하나를 공유하는 것은 어떻게 다릅니까? 예산을 넘었을 때 숫자를 올리기 전에 무엇을 물어야 합니까?

다음 04b에서는 이 계약과 예산을 그대로 두고 실패를 다룹니다. 실패 종류마다 다음 행동을 나누고, 병렬 Worker의 부분 실패를 합류에서 드러내고, 승인과 멱등을 그래프에 넣습니다.